# Part 1: K-Fold Stratification

Build the shared dataset, then compare regular KFold vs StratifiedKFold.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold

np.random.seed(42)

## Step 1: Build the dataset (used for the whole assignment)

Daily energy consumption with weather features.

In [2]:
n_days = 730  # 2 years of daily data

date_range = pd.date_range(start="2023-01-01", periods=n_days, freq="D")

day_of_year = date_range.dayofyear.values
day_of_week = date_range.dayofweek.values

# temperature follows a yearly seasonal pattern + noise
temperature = 20 + 10 * np.sin(2 * np.pi * (day_of_year - 80) / 365) + np.random.normal(0, 2, n_days)

# humidity, roughly inverse to temperature + noise
humidity = 70 - 0.5 * temperature + np.random.normal(0, 5, n_days)

# is_weekend flag
is_weekend = (day_of_week >= 5).astype(int)

# energy consumption (regression target)
# higher on weekdays, higher when temperature is very low or very high (heating/cooling)
energy = (
    50
    + 15 * (1 - is_weekend)
    + 0.8 * np.abs(temperature - 20)
    + 0.2 * humidity
    + np.random.normal(0, 5, n_days)
)

data = pd.DataFrame({
    "date": date_range,
    "day_of_week": day_of_week,
    "is_weekend": is_weekend,
    "temperature": temperature,
    "humidity": humidity,
    "energy": energy
})

# classification target: high_demand = 1 if energy above the 80th percentile, else 0
# using 80th percentile on purpose so the classes are IMBALANCED (80% vs 20%)
threshold = data["energy"].quantile(0.80)
data["high_demand"] = (data["energy"] > threshold).astype(int)

print("Dataset shape:\t", data.shape)
data.head()

Dataset shape:	 (730, 7)


,date,day_of_week,is_weekend,temperature,humidity,energy,high_demand
0,2023-01-01,6,1,11.214945,65.371754,69.410115,0
1,2023-01-02,0,0,9.982467,60.116903,78.915916,0
2,2023-01-03,1,0,11.594738,66.243895,83.927872,0
3,2023-01-04,2,0,13.388660,54.792752,76.995020,0
4,2023-01-05,3,0,9.920395,70.185580,84.198183,0


In [3]:
print("Class balance of high_demand target:")
print(data["high_demand"].value_counts())
print("Proportion:\t", round(data["high_demand"].mean(), 3), "are high_demand (class 1)")

# save for later parts (GRU, NN, subset selection, hyperparameter tuning)
data.to_csv("energy_dataset.csv", index=False)

Class balance of high_demand target:
high_demand
0    584
1    146
Name: count, dtype: int64
Proportion:	 0.2 are high_demand (class 1)


## Step 2: Regular KFold (no stratification)

In [4]:
X = data[["day_of_week", "is_weekend", "temperature", "humidity"]].values
y = data["high_demand"].values

k = 5

kf = KFold(n_splits=k, shuffle=True, random_state=42)

fold_num = 0
for train_idx, test_idx in kf.split(X):
    fold_num = fold_num + 1
    y_test_fold = y[test_idx]

    total_in_fold = len(y_test_fold)
    positives_in_fold = np.sum(y_test_fold)
    ratio = positives_in_fold / total_in_fold

    print("Fold", fold_num, "\ttest size:", total_in_fold, "\thigh_demand count:", positives_in_fold, "\tratio:", round(ratio, 3))

Fold 1 	test size: 146 	high_demand count: 30 	ratio: 0.205
Fold 2 	test size: 146 	high_demand count: 34 	ratio: 0.233
Fold 3 	test size: 146 	high_demand count: 25 	ratio: 0.171
Fold 4 	test size: 146 	high_demand count: 35 	ratio: 0.24
Fold 5 	test size: 146 	high_demand count: 22 	ratio: 0.151


## Step 3: Stratified K-Fold

In [5]:
skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

fold_num = 0
for train_idx, test_idx in skf.split(X, y):
    fold_num = fold_num + 1
    y_test_fold = y[test_idx]

    total_in_fold = len(y_test_fold)
    positives_in_fold = np.sum(y_test_fold)
    ratio = positives_in_fold / total_in_fold

    print("Fold", fold_num, "\ttest size:", total_in_fold, "\thigh_demand count:", positives_in_fold, "\tratio:", round(ratio, 3))

print("\nOverall dataset ratio was:\t", round(y.mean(), 3))
print("Notice: StratifiedKFold keeps every fold's ratio very close to this overall ratio.")
print("Regular KFold ratios can wander further away, especially with imbalanced classes.")

Fold 1 	test size: 146 	high_demand count: 29 	ratio: 0.199
Fold 2 	test size: 146 	high_demand count: 29 	ratio: 0.199
Fold 3 	test size: 146 	high_demand count: 29 	ratio: 0.199
Fold 4 	test size: 146 	high_demand count: 29 	ratio: 0.199
Fold 5 	test size: 146 	high_demand count: 30 	ratio: 0.205

Overall dataset ratio was:	 0.2
Notice: StratifiedKFold keeps every fold's ratio very close to this overall ratio.
Regular KFold ratios can wander further away, especially with imbalanced classes.
